## Context

Banks incur significant losses due to default in loans. This has led to a tightening up of loan underwriting and has increased loan rejection rates. The need for a better credit risk scoring model is also raised by banks.

The CNK bank has collected customer data for the past few years and wants to build a model to predict if a customer coming to purchase a loan is a good customer (will not default) or a bad customer (will default).

## Data Dictionary

- **Month** - the month of purchase
- **credit_amount** - amount for which loan is requested
- **credit_term** - for how long customer wants a loan
- **Age** - age of the customer
- **sex** - gender of the customer
- **education** - education level of customer
- **product_type** - for purchasing what type of product does the customer need a loan (0, 1, 2, 3, 4)
- **having_children_flg** - if the customer has children or not
- **region** - customer region category(0, 1, 2)
- **income** - income of the customer
- **family_status** - another, married, unmarried
- **phone_operator** - mobile operator category(0, 1, 2, 3)
- **is_client** - if the customer wanting to purchase a loan is our client or not
- **target** - 1-bad customer, 0-good customer

## Installing and Importing the Necessary Libraries

In [1]:
# Installing the libraries with specific versions
!pip install numpy==2.0.2 pandas==2.2.2 matplotlib==3.10.0 seaborn==0.13.2 scikit-learn==1.6.1 imbalanced-learn==0.13.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.4/238.4 kB 2.8 MB/s eta 0:00:00


**Note:**

- After running the above cell, kindly restart the notebook kernel (for Jupyter Notebook) or runtime (for Google Colab) and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [2]:
# To help with reading and manipulation of data
import numpy as np
from numpy import array
import pandas as pd

# To help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# To split the data
from sklearn.model_selection import train_test_split

# To impute missing values
from sklearn.impute import SimpleImputer

# To do one-hot encoding
from sklearn.preprocessing import OneHotEncoder

# To build a decision tree model
from sklearn.tree import DecisionTreeClassifier

# To get different performance metrics
import sklearn.metrics as metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    recall_score,
    accuracy_score,
    precision_score,
    f1_score,
)


# To undersample and oversample the data
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# To suppress warnings
import warnings
warnings.filterwarnings("ignore")

## Load and View the Dataset

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [17]:
#df = pd.read_csv("https://raw.githubusercontent.com/bootstrap0/adv-machine-learning/refs/heads/main/Loanclients.csv")
df = pd.read_csv("https://raw.githubusercontent.com/bootstrap0/adv-machine-learning/refs/heads/main/pima-indians-diabetes.csv")

In [28]:
data = df.copy()

In [59]:
import pandas as pd

# 1. Load the cardiac dataset
# Replace 'cardiac.csv' with your actual file path if it's different
df = pd.read_csv('https://raw.githubusercontent.com/bootstrap0/adv-machine-learning/refs/heads/main/Cardiac.csv')


df['UnderRisk'].value_counts(normalize = True)

,proportion
UnderRisk,
no,0.786277
yes,0.213723


In [61]:
## Encoding Existing and Attrited customers to 0 and 1 respectively, for analysis.

df["UnderRisk"].replace("yes", 1, inplace=True)

df["UnderRisk"].replace("no", 0, inplace=True)



X = df.drop(["UnderRisk"], axis=1)

y = df["UnderRisk"]



from sklearn.model_selection import train_test_split

# Splitting data into training, validation and test set:

# first we split data into 2 parts, say temporary and test



X_temp, X_test, y_temp, y_test = train_test_split(

    X, y, test_size=0.2, random_state=1, stratify=y

)

# then we split the temporary set into train and validation

X_train, X_val, y_train, y_val = train_test_split(

    X_temp, y_temp, test_size=0.25, random_state=1, stratify=y_temp

)

print(X_train.shape, X_val.shape, X_test.shape)



X_train = pd.get_dummies(X_train, drop_first=True)

X_val = pd.get_dummies(X_val, drop_first=True)

X_test = pd.get_dummies(X_test, drop_first=True)

print(X_train.shape, X_val.shape, X_test.shape)

(533, 12) (178, 12) (178, 12)
(533, 13) (178, 13) (178, 13)


In [62]:
from sklearn.linear_model import LogisticRegression

lr1 = LogisticRegression(random_state=1)

lr1.fit(X_train, y_train)

model_performance_classification_sklearn(lr1, X_train, y_train)



sm = SMOTE(

    sampling_strategy=1, k_neighbors=5, random_state=1

)  # Synthetic Minority Over Sampling Technique

X_train_over, y_train_over = sm.fit_resample(X_train, y_train)

lr2 = LogisticRegression(random_state=1)

lr2.fit(X_train_over, y_train_over)

model_performance_classification_sklearn(lr2, X_train_over, y_train_over)

NameError: name 'model_performance_classification_sklearn' is not defined

In [ ]:
from sklearn.ensemble import BaggingClassifier

bag = BaggingClassifier(random_state=1)

bag.fit(X_train_over, y_train_over)

model_performance_classification_sklearn(bag, X_val, y_val)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=1)

rf.fit(X_train, y_train)

confusion_matrix(y_train, rf.predict(X_train))

In [63]:
from sklearn.tree import DecisionTreeClassifier

from sklearn.model_selection import StratifiedKFold, cross_val_score

models = []  # Empty list to store all the models

# Appending models into the list

models.append(("Bagging", BaggingClassifier(random_state=1)))

models.append(("Random forest", RandomForestClassifier(random_state=1)))

models.append(("LR", LogisticRegression(random_state=1)))

models.append(("dtree", DecisionTreeClassifier(random_state=1)))

results = []  # Empty list to store all model's CV scores

names = []  # Empty list to store name of the models

# loop through all models to get the mean cross validated score

print("\n" "Cross-Validation Performance:" "\n")

for name, model in models:

    scoring = "recall"

    kfold = StratifiedKFold(

        n_splits=5, shuffle=True, random_state=1

    )  # Setting number of splits equal to 5

    cv_result = cross_val_score(

        estimator=model, X=X_train_over, y=y_train_over, scoring=scoring, cv=kfold

    )

    results.append(cv_result)

    names.append(name)

    print("{}: {}".format(name, cv_result.mean() * 100))


Cross-Validation Performance:



NameError: name 'X_train_over' is not defined

In [64]:
from sklearn.ensemble import AdaBoostClassifier

from sklearn import metrics

from sklearn.model_selection import RandomizedSearchCV



# defining model

model = AdaBoostClassifier(random_state=1)

# Parameter grid to pass in GridSearchCV

param_grid = {

    "n_estimators": np.arange(10, 110, 10),

    "learning_rate": [0.1, 0.01, 0.2, 0.05, 1],

    "base_estimator": [

        DecisionTreeClassifier(max_depth=1, random_state=1),

        DecisionTreeClassifier(max_depth=2, random_state=1),

        DecisionTreeClassifier(max_depth=3, random_state=1),

    ],

}

# Type of scoring used to compare parameter combinations

scorer = metrics.make_scorer(metrics.recall_score)

# Calling RandomizedSearchCV

randomized_cv = RandomizedSearchCV(

    estimator=model,

    param_distributions=param_grid,

    n_jobs=-1,

    n_iter=50,

    scoring=scorer,

    cv=5,

    random_state=1,

)

# Fitting parameters in RandomizedSearchCV

randomized_cv.fit(X_train_over, y_train_over)

print(

    "Best parameters are {} with CV score={}:".format(

        randomized_cv.best_params_, randomized_cv.best_score_

    )

)

NameError: name 'X_train_over' is not defined

In [65]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=1)

X_train_un, y_train_un = rus.fit_resample(X_train, y_train)

model1 = AdaBoostClassifier(random_state=1)

model1.fit(X_train_un, y_train_un)

model_performance_classification_sklearn(model1, X_train_un, y_train_un)

model2 = AdaBoostClassifier(random_state=1)

model2.fit(X_train_over, y_train_over)

model_performance_classification_sklearn(model2, X_train_over, y_train_over)

NameError: name 'model_performance_classification_sklearn' is not defined

In [ ]:
import matplotlib.pyplot as plt

model1 = AdaBoostClassifier(random_state=1)

model1.fit(X_train_un, y_train_un)

feature_names = X_train_un.columns

importances = model1.feature_importances_

indices = np.argsort(importances)

plt.figure(figsize=(12, 12))

plt.title("Feature Importances")

plt.barh(range(len(indices)), importances[indices], color="violet", align="center")

plt.yticks(range(len(indices)), [feature_names[i] for i in indices])

plt.xlabel("Relative Importance")

plt.show()